In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
import ot 
import scipy as sp
from risgw import risgw_gpu
from sgw_numpy import sgw_cpu
import torch
from SEINT.SEINT_numpy import SEINT

      

def RISGW(X, Y):
    n, p = np.shape(X)
    nproj1 =  round(np.log10(n)*10)
    device='cpu'
    dist = risgw_gpu(torch.from_numpy(X).to(torch.float32).to(device), torch.from_numpy(Y).to(torch.float32).to(device), device='cpu', nproj=nproj1)
    return dist

def GW(X, Y):
    C3 = sp.spatial.distance.cdist(X, X)
    C4 = sp.spatial.distance.cdist(Y, Y)
    C3 /= C3.max()
    C4 /= C4.max()
    p = ot.unif(len(X))
    q = ot.unif(len(Y))
    dist = ot.gromov.gromov_wasserstein(
        C3, C4, p, q, "square_loss", verbose=False, log=True)[1]["gw_dist"]
    return dist

def SEINT_Func(X, Y):
    n, p = np.shape(X)
    nproj1 =  round(np.log10(n)*10)
    dist = SEINT(X,Y,  maxed=True,set_seed = True, acc = True, rep = nproj1)
    return dist

def ISEINT(X, Y):
    n, p = np.shape(X)
    nproj1 =  round(np.log10(n)*10)
    dist = SEINT(X,Y, maxed=False, set_seed = True, acc = True, rep = nproj1)
    return dist
def sinkhorn(X, Y, reg=1e-5, p=2):
    a = ot.unif(len(X))         
    b = ot.unif(len(Y))
    M = ot.dist(X, Y, metric='euclidean') ** p  
    return ot.sinkhorn2(a, b, M, reg) ** (1.0 / p)


def emd(X, Y, p=2):
    a = ot.unif(len(X))
    b = ot.unif(len(Y))
    M = ot.dist(X, Y, metric='euclidean') ** p
    T = ot.emd(a, b, M)          
    dist = np.sum(T * M) ** (1.0 / p) 
    return dist


def SGW(X,Y):
    n, p = np.shape(X)
    nproj1 =  round(np.log10(n)*10)
    dist  = sgw_cpu(X,Y,nproj = nproj1)
    return dist 


METHODS = {
    "GW":     GW,
    "RISGW":      RISGW,
    "SGW":        SGW,
    "SINKHORN":        sinkhorn,
    "EMD": emd,
    "SEINT":         SEINT_Func,
    "ISEINT":  ISEINT,
}


def make_cov_matrices(theta: float, d: int):
    """
    Σ_X  = diag(3 I₂ , I_{d-2})
    Σ_Y  = diag(3 I₂ + 3 θ B₂ , I_{d-2}),  B₂ = [[0,1],[1,0]]
    """

    Sigma_X = np.eye(d)
    Sigma_X[:2, :2] = 3 * np.eye(2)

    B2 = np.array([[0., 1.],
                   [1., 0.]])
    Sigma_block = 3 * np.eye(2) + 3 * theta * B2
    Sigma_Y = np.eye(d)
    Sigma_Y[:2, :2] = Sigma_block
    return Sigma_X, Sigma_Y

def sample_gaussian(cov: np.ndarray, n: int,seed):
    np.random.seed(seed)
    return np.random.multivariate_normal(mean=np.zeros(cov.shape[0]), cov=cov, size=n)


def experiment_n_time(n_list, theta=0.1, d=4, n_trials=3):
    """
    Return:
        times_mean[method], times_std[method]  shape=(len(n_list),)
    """
    times_mean = {m: [] for m in METHODS}
    times_std  = {m: [] for m in METHODS}

    Σ_X_full, Σ_Y_full = make_cov_matrices(theta, d)  

    for n in n_list:
        trial_times = {m: [] for m in METHODS}
        # print(n)

        for _ in range(n_trials):
            # print(_)
            X = sample_gaussian(Σ_X_full, n,_*10)
            Y = sample_gaussian(Σ_Y_full, n,_*10)

            for name, func in METHODS.items():
                t0 = time.perf_counter()
                _  = func(X, Y)
                t1 = time.perf_counter()
                trial_times[name].append(t1 - t0)

        for name in METHODS:
            trial_times_arr = np.array(trial_times[name])
            times_mean[name].append(trial_times_arr.mean())
            times_std[name].append(trial_times_arr.std())

    return times_mean, times_std

In [ ]:
n_vals = (10 ** np.arange(1.0, 5.1, 0.5)).astype(int)
time_mean, time_std = experiment_n_time(n_vals, theta=0.5)

plt.figure(figsize=(6, 4))
log_n = np.log10(n_vals)
for name in METHODS:
    m  = np.array(time_mean[name])
    sd = np.array(time_std[name])
    plt.plot(log_n, np.log10(m), marker='o', label=name)          
    plt.fill_between(log_n, np.log10(m - sd), np.log10(m + sd), alpha=0.2)

plt.xlabel(r'$\log_{10}(n)$')
plt.ylabel(r'$\log_{10}(\text{time / sec})$')
plt.title('CPU Time versus $n$ (d=4, $\\theta=0$)')
plt.legend()
plt.tight_layout()
plt.show()